## Лабораторная работа №2. Обучение с учителем. Линейные параметрические модели.

### (1) IEEE-CIS Fraud Detection — задача бинарной классификации мошеннических онлайн-транзакций.

Во время выполнения лабораторной работы были использованы следующие сторонние источники:
1. Информация о файловой системе - https://yandex.cloud/ru/docs/datasphere/operations/data/dataset?ysclid=mmq4tdjpmi883592021.

### 1. Импорт библиотек

Нам понадобятся библиотеки для работы с файловой системой ВМ, загрузки, просмотра и обработки датасетов, а также обучения модели классификации.

In [1]:
# работа с файловой системой
import os

# чистка traceback
import warnings

# работа с датасетами как с массивами и таблицами
import numpy as np
import pandas as pd

# предварительная обработка данных в датасетах
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from feature_engine.encoding import CountFrequencyEncoder

# пользовательские трансформеры
from custom_transformers import OutlierClipper, CyclicalTimeEncoder

# готовые линейные модели из sklearn
from sklearn.linear_model import RidgeClassifier, LogisticRegression, SGDClassifier

# оценка точности предсказаний модели
from sklearn.metrics import accuracy_score, f1_score, classification_report

# перенос пайплайна fraud_pipeline.joblib и колонок fraud_processed_columns.joblib из ЛР1
import joblib

### 2. Загрузка и применение пайплайна предобработки данных из ЛР1

Сначала узнаем, как называются файлы в датасете, найдем нужный нам файл с данными для обучения и посмотрим, как он выглядит в общих чертах.

In [2]:
# путь к датасету согласно документации Yandex Cloud
dataset_path = "/home/jupyter/datasets/ieee-fraud-detection"

# список всех файлов в датасете
print("Файлы в датасете:")
print(os.listdir(dataset_path))

# путь к train_public
file_path = os.path.join(dataset_path, "train_public.csv")
# загрузка csv-файла
df = pd.read_csv(file_path)

# первые 5 строк
print("\nНабор тренировочных данных:")
df.head()

Файлы в датасете:
['train_public.csv', 'test_public.csv', 'sample_submission.csv']

Набор тренировочных данных:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


Разделим train_public на данные для обучения и для тестирования модели машинного обучения. Обработаем получившиеся два массива данных по отдельности пайплайном из ЛР1.

In [3]:
# чистка traceback, возникающего из-за вызова CountFrequencyEncoder из feature_engine.encoding
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.pipeline",
    message="This Pipeline instance is not fitted yet"
)

# постоянный сид для воспроизводимости результатов
seed = 8

# разделение train_public на признаки и целевую переменную isFraud
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

# разделение train_public на train data и test data
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=seed,
    stratify=y  # для несбалансированных данных в ieee-fraud-detection
)

# загрузка пайплайна и имен колонок
pipeline = joblib.load("fraud_pipeline.joblib")
processed_columns = joblib.load("fraud_processed_columns.joblib")

# предобработка данных пайплайном
X_tr_processed = pipeline.transform(X_tr)
X_te_processed = pipeline.transform(X_te)

### 3. Baseline-решение для классификации - модель, которая случайно угадывает метку класса

Реализация алгоритма, который случайно угадывает один из классов.

In [4]:
# классификатор, который случайным образом предсказывает классы и выдает равномерные вероятности для каждого класса
class RandomClassClassifier(BaseEstimator, ClassifierMixin):

    # конструктор класса (параметров нет)
    def __init__(self):
        pass
    
    # метод обучения модели
    def fit(self, X, y):
        # сохраняем уникальные классы из обучающей выборки
        self.classes_ = np.unique(np.asarray(y))
        # возвращаем self для совместимости с sklearn API
        return self
    
    # метод предсказания классов
    def predict(self, X):
        # для каждого объекта случайно выбираем один из известных классов
        return np.random.choice(self.classes_, size=len(X))
    
    # метод предсказания вероятностей классов
    def predict_proba(self, X):
        # n — количество объектов
        n = len(X)
        # k — количество классов
        k = len(self.classes_)
        # возвращаем матрицу вероятностей: для каждого объекта равномерное распределение по всем классам
        return np.full((n, k), 1.0 / k)

Обучим baseline-решение на train data, предскажем значения для test data и оценим результаты.

In [5]:
baseline_clf = RandomClassClassifier().fit(X_tr_processed, y_tr)
y_pred_base = baseline_clf.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_base))
print(classification_report(y_te, y_pred_base, target_names=["not fraud", "fraud"]))

accuracy: 0.5018415036830074
              precision    recall  f1-score   support

   not fraud       0.97      0.50      0.66     45584
       fraud       0.04      0.50      0.07      1660

    accuracy                           0.50     47244
   macro avg       0.50      0.50      0.36     47244
weighted avg       0.93      0.50      0.64     47244



Получившийся accuracy равен примерно 0.5, как и следовало ожидать от модели, выбирающей один из двух классов с 50% вероятностью.

### 4. Реализация PerceptronClassifier - линейного классификатора с ошибкой перцептрона

Напишем sklearn-совместимый класс PerceptronClassifier.<br>
Функция потерь - сумма отступов на неправильно классифицированных объектах. К функции потерь добавляется штраф.<br>
Метод минимизации функции потерь - стохастический градиентный спуск.

In [6]:
class PerceptronClassifier(BaseEstimator, ClassifierMixin):
    # передача параметров в конструкторе
    def __init__(self, learning_rate=0.001, n_iter=1000, batch_size=128, random_state=seed):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

    # переводим класс "0" в класс "-1" для margin
    def _prepare_y(self, y):
        return np.where(y == 1, 1, -1)

    # метод обучения модели
    def fit(self, X, y):
        # передача признаков объектов и таргета
        X = np.asarray(X)
        y = np.asarray(y)
        y = self._prepare_y(y)

        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)

        # инициализация весов
        self.w_ = rng.normal(scale=0.01, size=n_features)
        self.b_ = 0.0

        # n_iter раз проходимся по всем объектам обучающей выборки
        for epoch in range(self.n_iter):
            # перемешивание объектов датасета
            indices = rng.permutation(n_samples)

            # проходимся по всем объектам обучающей выборки
            for start in range(0, n_samples, self.batch_size):
                # разбивка на пакеты
                batch_idx = indices[start:start + self.batch_size]
                Xb = X[batch_idx]
                yb = y[batch_idx]

                # векторизация из-за пакетов
                scores = Xb @ self.w_ + self.b_ # решающая функция
                margins = yb * scores # отступы как основа функции потерь

                # ошибочная классификация
                mask = margins <= 0
                
                # инициализация градиентов
                grad_w = np.zeros(n_features)
                grad_b = 0.0

                # обновление градиента по ошибочным объектам (усреднение по батчу)
                if np.any(mask):
                    grad_w -= np.sum((yb[mask][:, None] * Xb[mask]), axis=0) / len(yb) # средний по батчу градиент по переменным весов
                    grad_b -= np.sum(yb[mask]) / len(yb) # средний по батчу градиент по переменной смещения, чтобы не приходилось ставить очень маленький learning_rate

                # шаг в сторону антиградиента
                self.w_ -= self.learning_rate * grad_w
                self.b_ -= self.learning_rate * grad_b
                
            # отслеживание прогресса обучения
            if epoch%100==0:
                print("number of epoch:", epoch)

        return self

    # найденная решающая функция fw(X) = <x, w> + b
    def decision_function(self, X):
        X = np.asarray(X)
        return X @ self.w_ + self.b_

    # распределение объектов по классам в зависимости от их положения относительно fw(X) = 0
    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, 0)

    # вероятность принадлежности объектов к классу 0 и к классу 1
    def predict_proba(self, X):
        scores = self.decision_function(X)
        probs = 1 / (1 + np.exp(-scores))
        return np.vstack([1 - probs, probs]).T

Обучим Perceptron Classifier на train data, предскажем значения для test data и оценим результаты.

In [7]:
model_pc = PerceptronClassifier(learning_rate=0.001, n_iter=1000, batch_size=128, random_state=seed).fit(X_tr_processed, y_tr)
y_pred_pc = model_pc.predict(X_te_processed)

print("accuracy:", accuracy_score(y_te, y_pred_pc))
print(classification_report(y_te, y_pred_pc, target_names=["not fraud", "fraud"]))

number of epoch: 0
number of epoch: 100
number of epoch: 200
number of epoch: 300
number of epoch: 400
number of epoch: 500
number of epoch: 600
number of epoch: 700
number of epoch: 800
number of epoch: 900
accuracy: 0.9659004318008636
              precision    recall  f1-score   support

   not fraud       0.97      1.00      0.98     45584
       fraud       0.74      0.05      0.09      1660

    accuracy                           0.97     47244
   macro avg       0.85      0.52      0.53     47244
weighted avg       0.96      0.97      0.95     47244



Учитывая отсутствие классовых весов и прочего тюнинга при таком сильном дисбалансе классов, f1-score = 0.98 для распространенного класса и f1-score = 0.09 для редкого класса - ожидаемый результат для простой ошибки перцептрона без L2-регуляризации.

### 5. Серия экспериментов с готовыми реализациями линейных моделей из sklearn

Чтобы сформировать предсказания для test_public, подберем линейную модель, которая может сделать это достаточно хорошо.<br>
Для этого обучим с различными параметрами несколько готовых линейных моделей из sklearn и оценим их accuracy, precision, recall и f1-score.

In [8]:
# пример обучения готовой модели и сбора предсказаний от нее
model_sgdc = SGDClassifier(loss='hinge', alpha=0.0001, tol=0.001, random_state=seed, learning_rate='invscaling', eta0=12, class_weight=None).fit(X_tr_processed, y_tr)
y_pred_sgdc = model_sgdc.predict(X_te_processed)

# метрики
print("Эксперимент №27:")
print("\naccuracy:", accuracy_score(y_te, y_pred_sgdc))
print(classification_report(y_te, y_pred_sgdc, target_names=["not fraud", "fraud"]))

Эксперимент №27:

accuracy: 0.9663660993988654
              precision    recall  f1-score   support

   not fraud       0.97      0.99      0.98     45584
       fraud       0.54      0.26      0.35      1660

    accuracy                           0.97     47244
   macro avg       0.76      0.63      0.67     47244
weighted avg       0.96      0.97      0.96     47244



In [9]:
# сведение результатов экспериментов с их интерпретацией в электронную таблицу и ее сохранение
experiment_results = "https://docs.google.com/spreadsheets/d/1NJWK0HEngAb6L46L2Ldg5_3LpSeoFWHGN5EprWrgCF4/export?format=csv"
df_exp = pd.read_csv(experiment_results)
df_exp.to_csv(
    '/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/fraud_experiments_results.csv', 
    index=False
)
print(f"Таблица экспериментов сохранена в fraud_experiments_results.csv")

Таблица экспериментов сохранена в fraud_experiments_results.csv


Результаты экспериментов, сведенные в таблицу: https://docs.google.com/spreadsheets/d/1NJWK0HEngAb6L46L2Ldg5_3LpSeoFWHGN5EprWrgCF4/edit?gid=0#gid=0<br>
Из таблицы видно, что даже готовые линейные модели из sklearn выдают очень плохие результаты для крайне редкого класса fraud, какие бы параметры мы ни подбирали, но это было довольно ожидаемо. Случайным образом удалось выбить у SGD f1-score = 0.35 при precision = 0.54 и recall = 0.26. В остальных случаях при высоком precision очень низкий recall и наоборот, а вместе они дают f1-score в районе 0.25.<br>
Таким образом, одним подбором параметров от линейных моделей тяжело получить достаточно хорошие результаты классификации. Тогда подберем threshold для 4 лучших моделей из таблицы ради увеличения f1-score у fraud, считая, что нам важно не только распознать fraud, но и не обвинить not fraud (иначе можно было бы просто вызвать любую модель с высоким recall у fraud). Среди этих 4 найдем модель с самым высоким f1-score, чтобы она дала предсказания для test_public.

In [10]:
# поиск лучшего порога
def find_best_threshold(model, X, y, thresholds=None):
    
    # получаем вероятности или просто решающую функцию
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X)[:, 1]
    else:
        scores = model.decision_function(X)
    
    # перебираем пороги
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.5, 100)
    
    best_t = 0
    best_f1 = 0
    
    for t in thresholds:
        preds = (scores > t).astype(int)
        f1 = f1_score(y, preds)
        
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    
    return best_t, best_f1

Ridge Classifier:

In [11]:
# чистка traceback, возникающего из-за очень большой размерности train data
warnings.filterwarnings("ignore", category=RuntimeWarning)

model_rc = RidgeClassifier(alpha=0.001, max_iter=None, tol=0.0001, class_weight=None, random_state=seed).fit(X_tr_processed, y_tr)

trc, f1rc = find_best_threshold(model_rc, X_te_processed, y_te)
print("best threshold:", trc, "\nf1-score for fraud:", f1rc)

best threshold: 0.01 
f1-score for fraud: 0.26899383983572894


Logistic Regression:

In [12]:
model_lr = LogisticRegression(C=1.0, tol=0.0001, class_weight=None, random_state=seed, solver='lbfgs', max_iter=1000).fit(X_tr_processed, y_tr)

tlr, f1lr = find_best_threshold(model_lr, X_te_processed, y_te)
print("best threshold:", tlr, "\nf1-score for fraud:", f1lr)

best threshold: 0.1782828282828283 
f1-score for fraud: 0.4018506278916061


SGD Classifier (loss - log loss):

In [13]:
model_sgdc_log = SGDClassifier(loss='log_loss', alpha=0.0001, max_iter=1000, tol=0.001, random_state=seed, learning_rate='optimal', class_weight=None).fit(X_tr_processed, y_tr)

tsgdc_log, f1sgdc_log = find_best_threshold(model_sgdc_log, X_te_processed, y_te)
print("best threshold:", tsgdc_log, "\nf1-score for fraud:", f1sgdc_log)

best threshold: 0.21787878787878787 
f1-score for fraud: 0.32007400555041626


SGD Classifier (loss - modified huber):

In [14]:
model_sgdc_modified = SGDClassifier(loss='modified_huber', alpha=0.0001, max_iter=1000, tol=0.001, random_state=seed, learning_rate='optimal', class_weight=None).fit(X_tr_processed, y_tr)

tsgdc_modified, f1sgdc_modified = find_best_threshold(model_sgdc_modified, X_te_processed, y_te)
print("best threshold:", tsgdc_modified, "\nf1-score for fraud:", f1sgdc_modified)

best threshold: 0.4703030303030303 
f1-score for fraud: 0.27553159628631324


Самый лучший результат у Logistic Regression - f1-score = 0.40 для fraud при threshold = 0.178. Теперь посмотрим на остальные метрики с этим порогом.

In [15]:
# вероятности fraud
y_proba_lr = model_lr.predict_proba(X_te_processed)[:, 1]

# порог
threshold = 0.1782828282828283

# предсказания по порогу
y_pred_lr = (y_proba_lr >= threshold).astype(int)

# метрики
print("accuracy:", accuracy_score(y_te, y_pred_lr))
print(classification_report(y_te, y_pred_lr, target_names=["not fraud", "fraud"]))

accuracy: 0.9616882567098467
              precision    recall  f1-score   support

   not fraud       0.98      0.98      0.98     45584
       fraud       0.45      0.37      0.40      1660

    accuracy                           0.96     47244
   macro avg       0.71      0.67      0.69     47244
weighted avg       0.96      0.96      0.96     47244



Общий accuracy и f1-score для not fraud остались очень высокими, а precision и recall для fraud имеют относительно сбалансированные значения. Тогда возьмем именно эту модель машинного обучения для формирования предсказаний.

### 6. Создание таблицы с предсказаниями для test_public.csv

Итак, выбранная модель - Logistic Regression. Дадим предсказания для test public и сохраним их в csv-файл.

In [16]:
# путь к test_public
file_test_path = os.path.join(dataset_path, "test_public.csv")
# загрузка csv-файла
df_test = pd.read_csv(file_test_path)

# предобработка данных пайплайном
X_test_processed = pipeline.transform(df_test)

# вероятности fraud
test_proba_lr = model_lr.predict_proba(X_test_processed)[:, 1]

# формирование итоговой таблицы
result = pd.DataFrame({
    "TransactionID": df_test["TransactionID"],
    "isFraud": test_proba_lr
})

# сохранение
output_path = os.path.join("/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/", "classification_submission.csv")
result.to_csv(output_path, index=False)

print("Сохранено в:", output_path)

Сохранено в: /home/jupyter/project/ml-2026-spring-hw-02-superkroshki/classification_submission.csv


### (2) NYC Taxi - задача регрессии, связанная с предсказанием длительности поездки по табличным признакам.

Результат серии экспериментов находится в файле taxi_experiments_results.csv 
и по ссылке: https://docs.google.com/spreadsheets/d/1X24vdjcZLSDEa1EdOftEhVOy6wWML-CqHKlqdvsV0yA/edit?usp=sharing

### 1. Импорт библиотек

In [17]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, RegressorMixin,TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

### 2. Чтение из файла

In [18]:
# Загрузка данных
x = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/X_train_public.csv")
y = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/y_train_public.csv")

### 3. Добавление кастомных трансформеров для нормальной работы пайплайна

In [19]:
class DateTimeExtractor(BaseEstimator, TransformerMixin):
    
    # Извлекает временные признаки из datetime столбца 'tpep_pickup_datetime'
    # Создает циклические признаки для времени суток (sin/cos) и календарные признаки
    
    def __init__(self):
        self.seconds_in_day = 24 * 60 * 60      # Секунд в сутках
        self.seconds_in_hour = 3600             # Секунд в часе
        self.seconds_in_minute = 60             # Секунд в минуте
        self.feature_names_in_ = None
    
    def fit(self, X, y=None):
    # Сохраняет имена входных признаков для get_feature_names_out
        if hasattr(X, 'columns'):
            self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
    # Преобразует datetime в признаки:
    #   - pickup_dayofweek: день недели (0=понедельник, 6=воскресенье)
    #   - pickup_hour: час pickup
    #   - pickup_month: месяц
    #   - pickup_day: день месяца
    #   - time_sin, time_cos: циклические признаки времени суток
        
    # Циклические признаки кодируют время непрерывно (учитывая переход через полночь):
    # time_of_day = (hour*3600 + minute*60 + second) / 86400
    # time_rad = 2π * time_of_day

        X_new = X.copy()
        X_new['tpep_pickup_datetime'] = pd.to_datetime(X_new['tpep_pickup_datetime'])
        
        # Календарные признаки
        X_new['pickup_dayofweek'] = X_new['tpep_pickup_datetime'].dt.dayofweek
        X_new['pickup_hour'] = X_new['tpep_pickup_datetime'].dt.hour
        X_new['pickup_month'] = X_new['tpep_pickup_datetime'].dt.month
        X_new['pickup_day'] = X_new['tpep_pickup_datetime'].dt.day
        
        # Циклическое кодирование времени суток
        time_of_day = (X_new['tpep_pickup_datetime'].dt.hour * self.seconds_in_hour +
                      X_new['tpep_pickup_datetime'].dt.minute * self.seconds_in_minute +
                      X_new['tpep_pickup_datetime'].dt.second) / self.seconds_in_day
        time_of_day_rad = 2 * np.pi * time_of_day
        X_new['time_sin'] = np.sin(time_of_day_rad)
        X_new['time_cos'] = np.cos(time_of_day_rad)
        
        return X_new.drop(columns=['tpep_pickup_datetime'])
    
    def get_feature_names_out(self, input_features=None):
    # Возвращает имена выходных колонок
        if input_features is None:
            original_features = [col for col in self.feature_names_in_
                                if col not in ['tpep_pickup_datetime']]
        else:
            original_features = [col for col in input_features
                                if col not in ['tpep_pickup_datetime']]
        return original_features + ['pickup_dayofweek', 'pickup_hour', 
                                   'pickup_month', 'pickup_day', 'time_sin', 'time_cos']

In [20]:
class LocationFrequencyEncoder(BaseEstimator, TransformerMixin):
    # Кодирует частоты локаций и пар локаций
    # Заменяет категориальные PULocationID/DOLocationID на их частоты встречаемости
    def __init__(self):
        self.location_freq_ = None      # Частоты отдельных локаций
        self.pair_freq_ = None          # Частоты пар (PU_DO)

    def fit(self, X, y=None):
    # Вычисляет частоты локаций и пар локаций на тренировочных данных
        # Частота каждой локации (PU и DO вместе)
        all_locations = pd.concat([X['PULocationID'], X['DOLocationID']])
        self.location_freq_ = all_locations.value_counts(normalize=True).to_dict()

        # Частота каждой пары (PU_DO)
        location_pairs = (X['PULocationID'].astype(str) + '_' + 
                         X['DOLocationID'].astype(str))
        self.pair_freq_ = location_pairs.value_counts(normalize=True).to_dict()

        if hasattr(X, 'columns'):
            self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
    # Преобразует локации в их частоты
    # Создает признаки: PU_freq, DO_freq, location_pair_freq

        X_new = X.copy()
        X_new['PU_freq'] = X_new['PULocationID'].map(self.location_freq_).fillna(0)
        X_new['DO_freq'] = X_new['DOLocationID'].map(self.location_freq_).fillna(0)
        
        location_pair = (X_new['PULocationID'].astype(str) + '_' + 
                        X_new['DOLocationID'].astype(str))
        X_new['location_pair_freq'] = location_pair.map(self.pair_freq_).fillna(0)
        
        return X_new.drop(columns=['PULocationID', 'DOLocationID'])
    
    def get_feature_names_out(self, input_features=None):
    # Возвращает имена выходных признаков
        if input_features is None:
            original_features = [col for col in self.feature_names_in_
                                if col not in ['PULocationID', 'DOLocationID']]
        else:
            original_features = [col for col in input_features
                                if col not in ['PULocationID', 'DOLocationID']]
        return original_features + ['PU_freq', 'DO_freq', 'location_pair_freq']



In [21]:
class OutlierClipper(BaseEstimator, TransformerMixin):
    # Обрезает выбросы по правилу IQR (1.0 * IQR)
    # Для каждого числового признака: [Q1-1.0*IQR, Q3+1.0*IQR]
    def fit(self, X, y=None):
        # Вычисляет параметры для обрезки выбросов (Q1, Q3, IQR) для каждого признака

        self.q1 = np.nanpercentile(X, 25, axis=0)      # 25-й процентиль
        self.q3 = np.nanpercentile(X, 75, axis=0)      # 75-й процентиль
        self.iqr = self.q3 - self.q1                    # Межквартильный размах
        
        if hasattr(X, 'columns'):
            self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
        # Обрезает выбросы в данных по вычисленным границам
        # Пропускает признаки с IQR=0 (константы)

        X_new = X.copy()
        for i, col in enumerate(X_new.columns):
            if self.iqr[i] == 0:  # Пропускаем константные признаки
                continue
            lower = self.q1[i] - 1.0 * self.iqr[i]    # Нижняя граница
            upper = self.q3[i] + 1.0 * self.iqr[i]    # Верхняя граница
            X_new[col] = np.clip(X_new[col], lower, upper)
        return X_new
       
    def inverse_transform(self, X):
        # Возвращает данные без изменений 
        return X
    
    def get_feature_names_out(self, input_features=None):
        # Возвращает те же имена признаков (не создаем новые)
        if input_features is not None:
            return input_features
        return getattr(self, 'feature_names_in_', None)

# Создание экземпляра для числовых выбросов
numeric_outlier = OutlierClipper()

### 4. Применение пайплайна к обучающей выборке

In [22]:
# Подготовка данных
# Разделение выборки на обучающую и валидационную
y = pd.DataFrame(y, columns=['trip_duration_sec'])
train_features, val_features, train_target, val_target = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [23]:
# Загрузка пайплайнов
pipe_x = joblib.load('/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/pipe_x.joblib')
pipe_y = joblib.load('/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/pipe_y.joblib')

In [24]:
# Применение пайпланов к данным
train_features_tr = pipe_x.transform(train_features)
val_features_tr = pipe_x.transform(val_features)
train_target_tr = pipe_y.transform(train_target)
val_target_tr = pipe_y.transform(val_target)

### 5. Создание Baseline-модели

In [25]:
class MeanRegressor(BaseEstimator, RegressorMixin):
    # Базовый регрессор, предсказывающий среднее значение целевого признака
    # Используется как baseline для сравнения с другими моделями
    def __init__(self):
        self.y_mean = None
        
    def fit(self, X, y):
    # Обучает модель, вычисляя среднее значение целевого признака
        self.y_mean = float(np.mean(y))
        print(f'Среднее значение: {self.y_mean}')
        return self
    def predict(self, X):
    # Предсказывает среднее значение для всех объектов
        return np.full(shape=(len(X),), fill_value=self.y_mean, dtype=float)


In [26]:
# Тестирование baseline
baseline_reg = MeanRegressor().fit(train_features_tr, train_target_tr)
yr_pred = baseline_reg.predict(val_features_tr)

print("Baseline регрессор, среднее значение:")
print("RMSE:", mean_squared_error(val_target_tr, yr_pred))
print("MAE :", mean_absolute_error(val_target_tr, yr_pred))
print("R^2 :", round(r2_score(val_target_tr, yr_pred), 4))

Среднее значение: 846.9100879778674
Baseline регрессор, среднее значение:
RMSE: 264663.7423411019
MAE : 423.8081472257199
R^2 : -0.0


### 6. Реализация LinnearRegressionOLS - аналитическое неитерационное решение регрессии

In [27]:
class LinRegression(BaseEstimator, RegressorMixin):
    # Линейная регрессия с аналитическим решением через нормальное уравнение
    # Поддерживает L2-регуляризацию для предотвращения переобучения
    def __init__(self, alpha=1.0):  # alpha = lambda (коэффициент L2-регуляризации)
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
    # Обучает модель аналитически через нормальное уравнение с L2-регуляризацией
    # Для устойчивости к вырожденности матрицы X^T @ X используется L2-регуляризация:
    # w = (X^T @ X + alpha * I)^(-1) @ X^T @ y

    # Если alpha > 0, матрица всегда невырожденная. Для alpha = 0 используется
    # псевдообратная матрица np.linalg.pinv для обработки вырожденных случаев
        
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, (pd.Series, pd.DataFrame)):
            y = y.values.ravel() if isinstance(y, pd.DataFrame) else y.values
            
        X_b = np.c_[np.ones(X.shape[0]), X]  # Добавляем столбец единиц для свободного члена
        n_features = X_b.shape[1]
        I = np.eye(n_features)
        I[0, 0] = 0  # Не регуляризуем свободный член (intercept)
        
        # Вычисляем веса
        if self.alpha == 0:
            # Используем псевдообратную матрицу для вырожденных случаев
            w = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        else:
            # Используем L2-регуляризацию для устойчивости
            XtX = X_b.T @ X_b
            regularized_matrix = XtX + self.alpha * I
            w = np.linalg.inv(regularized_matrix) @ X_b.T @ y
        
        self.intercept_ = w[0]
        self.coef_ = w[1:]
        return self
    
    def predict(self, X):
    # Предсказывает значения для новых данных
        if isinstance(X, pd.DataFrame):
            X = X.values
        return X @ self.coef_ + self.intercept_

In [28]:
# Тестирование собственной линейной регрессии
lin_reg = LinRegression().fit(train_features_tr, train_target_tr)
yr_pred = lin_reg.predict(val_features_tr)

print("\nЛинейная  регрессия :")
print("RMSE:", mean_squared_error(val_target_tr, yr_pred))
print("MAE :", mean_absolute_error(val_target_tr, yr_pred))
print("R^2 :", round(r2_score(val_target_tr, yr_pred), 8))


Линейная  регрессия :
RMSE: 43487.42503678754
MAE : 137.31713382582498
R^2 : 0.83568798


### 7. Серия экспериментов с готовыми реализациями моделей

In [29]:
# Серия экспериментов с различными моделями
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
models = {
    'Lin': LinearRegression(),
    'Ridge': GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10]}, cv=5),
    'Lasso': GridSearchCV(Lasso(), {'alpha': [0.01, 0.1, 1]}, cv=5),
    'ElasticNet': GridSearchCV(ElasticNet(), {'alpha': [0.01, 0.1], 'l1_ratio': [0.5, 0.7, 0.9]}, cv=5)
}

# Автоматическое сохранение таблицы экспериментов
experiments_results = []

best_score = -np.inf
best_model = None
best_name = None

for name, model in models.items():
    print(f"\nОбучение модели {name}...")
    model.fit(train_features_tr, train_target_tr)
    pred = model.predict(val_features_tr)
    
    mse = mean_squared_error(val_target_tr, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(val_target_tr, pred)
    score = round(r2_score(val_target_tr, pred), 8)
    
    # Сохраняем результаты эксперимента
    experiments_results.append({
        'model': name,
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'R2': score
    })
    
    print(f"{name}: R² = {score:.8f}, RMSE = {rmse:.4f}")
    if name != 'Lin':
        print(f"Лучшие параметры: {model.best_params_}")
    
    if score > best_score:
        best_score = score
        best_model = model
        best_name = name



Обучение модели Lin...
Lin: R² = 0.83585799, RMSE = 208.4285

Обучение модели Ridge...
Ridge: R² = 0.83585802, RMSE = 208.4285
Лучшие параметры: {'alpha': 0.01}

Обучение модели Lasso...
Lasso: R² = 0.83487478, RMSE = 209.0518
Лучшие параметры: {'alpha': 0.01}

Обучение модели ElasticNet...
ElasticNet: R² = 0.83174357, RMSE = 211.0246
Лучшие параметры: {'alpha': 0.01, 'l1_ratio': 0.9}


In [30]:
# Сохранение таблицы экспериментов
experiments_df = pd.DataFrame(experiments_results)
experiments_df.to_csv(
    '/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/taxi_experiments_results.csv', 
    index=False
)
print(f"\nТаблица экспериментов сохранена в experiments_results.csv")
print(experiments_df)

print(f"\nЛучшая модель: {best_name} с R² = {best_score:.8f}")


Таблица экспериментов сохранена в experiments_results.csv
        model         MSE      RMSE       MAE        R2
0         Lin  43442.4300  208.4285  137.6350  0.835858
1       Ridge  43442.4213  208.4285  137.6288  0.835858
2       Lasso  43702.6510  209.0518  137.4199  0.834875
3  ElasticNet  44531.3680  211.0246  139.5038  0.831744

Лучшая модель: Ridge с R² = 0.83585802


Наилучший результат у Ridge регрессии с гиперпараметром alpha = 0.01. R^2 = 0.835

### 8. Создание файла с предсказаниями trip_duration.csv

In [31]:
# Генерация предсказаний для тестовой выборки
print("\nГенерация предсказаний для тестовой выборки...")
x_test = pd.read_csv("/home/jupyter/datasets/nyc-taxi-small-public/X_test_public.csv")
x_test = pipe_x.transform(x_test)
y_test = best_model.predict(x_test).ravel()


Генерация предсказаний для тестовой выборки...


In [32]:
# Создание и сохранение файла предсказаний 
submission = pd.DataFrame({
    "row_id": range(len(y_test)), 
    "trip_duration_sec": y_test
})

In [33]:
submission.to_csv(
    "/home/jupyter/project/ml-2026-spring-hw-02-superkroshki/regression_submission.csv", 
    index=False
)
print("Файл предсказаний сохранен: regression_submission.csv")
print(f"Размер submission: {submission.shape}")
print(submission.head())

Файл предсказаний сохранен: regression_submission.csv
Размер submission: (3513200, 2)
   row_id  trip_duration_sec
0       0        1655.683118
1       1        1193.009415
2       2        1143.918223
3       3         462.761518
4       4         265.918259
